<a href="https://colab.research.google.com/github/VikaSvyat/DI_Bootcamp/blob/main/Week14/BERT_SentimentAssistant_W14D1_MP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Mini Project: Sentiment Assistant with BERT Fine-Tuning

https://octopus.developers.institute/courses/collection/124/course/725/section/1980/chapter/4741

In [ ]:
!pip install -q tensorflow tensorflow-datasets accelerate evaluate
!pip install "transformers==4.41.2"

In [2]:
import platform
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BertTokenizer, TFBertForSequenceClassification

print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices('GPU'))

Python version      : 3.12.13
TensorFlow version  : 2.19.0
GPU devices detected: []


In [3]:
import transformers
print(transformers.__version__)
print(transformers.__file__)

4.41.2
/usr/local/lib/python3.12/dist-packages/transformers/__init__.py


Load the IMDB Reviews Dataset

In [ ]:
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,
    with_info=True
)
print(ds_info)

In [6]:
##samples to make sentiment concrete
for text, label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() else "Negative")
    print(text.numpy().decode()[:250], "...\n")

Label: Negative
This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline ...

Label: Negative
I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However on this occasion I fell asleep because the film wa ...



Tokenizer Setup & Data Pipeline

In [7]:
####Config

In [ ]:
MAX_LENGTH = 256   # trim or pad every review to 256 tokens so batches align
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer loaded:", tokenizer.name_or_path)

convert raw bytes → token IDs, attention masks, and segment IDs.

In [9]:
# def encode_review(review_input):
#     if isinstance(review_input, bytes):
#         review_text = review_input.decode("utf-8")
#     elif hasattr(review_input, "numpy"):
#         review_text = review_input.numpy().decode("utf-8")
#     else:
#         review_text = str(review_input)

#     return tokenizer.encode_plus(
#         review_text,
#         add_special_tokens=True,
#         max_length=MAX_LENGTH,
#         padding="max_length",
#         truncation=True,
#         return_attention_mask=True,
#         return_token_type_ids=True,
#     )

# def tf_encode(text, label):
#     encoded = tf.py_function(
#         func=lambda t: list(encode_review(t).values()),
#         inp=[text],
#         Tout=[tf.int32, tf.int32, tf.int32]
#     )
#     return {
#         "input_ids": encoded[0],
#         "attention_mask": encoded[1],
#         "token_type_ids": encoded[2]
#     }, label

# def prepare_dataset(dataset):
#     return (
#         dataset
#         .map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)
#         .shuffle(2000)
#         .batch(BATCH_SIZE)
#         .prefetch(tf.data.AUTOTUNE)
#     )

##### Tokenization
def tokenize(text, label):
    text = tf.strings.reduce_join(text)

    text = text.numpy().decode("utf-8")

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

    return (
        tokens["input_ids"],
        tokens["attention_mask"],
        tokens["token_type_ids"],
        label
    )
####
def tf_tokenize(text, label):
    input_ids, attention_mask, token_type_ids, label = tf.py_function(
        func=tokenize,
        inp=[text, label],
        Tout=[tf.int32, tf.int32, tf.int32, tf.int64]
    )

    input_ids.set_shape([MAX_LENGTH])
    attention_mask.set_shape([MAX_LENGTH])
    token_type_ids.set_shape([MAX_LENGTH])

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "token_type_ids": token_type_ids,
    }, label


### Dataset pipeline
def prepare_dataset(ds, shuffle=False):
    ds = ds.map(tf_tokenize, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(2000)

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

#### Train / Val / Test split
val_size = int(0.1 * len(ds_train))

ds_val = ds_train.take(val_size)
ds_train = ds_train.skip(val_size)

train_ds = prepare_dataset(ds_train, shuffle=True)
val_ds   = prepare_dataset(ds_val)
test_ds  = prepare_dataset(ds_test)

print("Train dataset size:", len(ds_train))
print("Val   dataset size:", len(ds_val))
print("Test  dataset size:", len(ds_test))

Train dataset size: 22500
Val   dataset size: 2500
Test  dataset size: 25000


Initialize the Fine-Tuning Model

In [ ]:
#### Model
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False
)
#### Compile
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()


In [ ]:
##### Train
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3
)

In [ ]:
##### Test evaluation
eval_metrics = model.evaluate(test_ds)

**What lever most improved results?**


The biggest improvement usually comes from data quality and preprocessing, not just model tweaks. Cleaning the text (removing noise, fixing labels, balancing classes) has a larger impact than simply increasing epochs or slightly tuning hyperparameters.

Data cleaning → biggest gain (better signal)


Hyperparameters (learning rate, batch size) → moderate improvement
More epochs → limited benefit, can even cause overfitting

**Where would you add guardrails before deployment?**

Before using this sentiment model in production, guardrails are essential:

- Confidence threshold

Only act on predictions when confidence is high (e.g. > 0.7)

- Human-in-the-loop

Route low-confidence or high-impact cases to human review

- Bias monitoring

Check performance across different user groups or language styles

- Input validation

Reject or flag:

-very short inputs

-non-English text (if model trained on English)

-irrelevant content

- Drift detection

Monitor if real-world data starts differing from training data

**Which stakeholders benefit the most?**
- Support Lead

Identifies unhappy customers quickly
Prioritizes urgent tickets
Improves response time and satisfaction

- Product Manager

Gains insights into user pain points
Tracks sentiment trends over time
Guides feature improvements

- Compliance Officer

Detects risky or sensitive interactions
Flags potential escalation cases
Ensures communication standards